# Reproducing NIST SP 1065

W. J. Riley's *Handbook of Frequency Stability Analysis* (NIST Special Publication 1065, 2008,
section 12) publishes test suites so that implementations can be checked. This notebook
regenerates the data and compares ntpstats with the printed values:

* **Table 30**: the nine-point frequency data of NBS Monograph 140, Annex 8.E;
* **Table 31**: the 1000-point suite from the generator `n[i+1] = 16807 n[i] mod 2147483647`.

The tables print seven significant digits; the same comparison runs in the test suite
(`tests/test_reference_nist.py`).

<sub>MIT licensed, part of [ntpstats](https://github.com/thiagodefreitas/NetworkTime). Runs in CI on every change.</sub>

In [ ]:
from pathlib import Path

import numpy as np

from ntpstats import api as nt

# examples/data, whether the notebook runs from examples/notebooks or the repository root
DATA = next(p for p in (Path("../data"), Path("examples/data"), Path("../../examples/data")) if p.is_dir())

NBS9 = [892, 809, 823, 798, 671, 644, 883, 903, 677]
TABLE30 = {  # averaging factors 1, 2
    "adev": [91.22945, 115.8082], "oadev": [91.22945, 85.95287], "mdev": [91.22945, 74.78849],
    "tdev": [52.67135, 86.35831], "hdev": [70.80607, 85.61487], "totdev": [91.22945, 93.90379],
    "mtot": [75.50203, 75.83606], "ttot": [43.59112, 87.56794],
}
TABLE31 = {  # averaging factors 1, 10, 100
    "adev": [2.922319e-01, 9.965736e-02, 3.897804e-02], "oadev": [2.922319e-01, 9.159953e-02, 3.241343e-02],
    "mdev": [2.922319e-01, 6.172376e-02, 2.170921e-02], "tdev": [1.687202e-01, 3.563623e-01, 1.253382e00],
    "hdev": [2.943883e-01, 9.581083e-02, 3.237638e-02], "totdev": [2.922319e-01, 9.134743e-02, 3.406530e-02],
    "mtot": [2.418528e-01, 6.499161e-02, 2.287774e-02], "ttot": [1.396338e-01, 3.752293e-01, 1.320847e00],
}


def nist1000():
    n, out = 1234567890, []
    for _ in range(1000):
        out.append(n / 2147483647)
        n = (16807 * n) % 2147483647
    return np.array(out)


def to_phase(y):
    """Frequency data to phase (tau0 = 1); these statistics do not depend on the mean."""
    y = np.asarray(y, dtype=float)
    return np.concatenate(([0.0], np.cumsum(y - y.mean())))

## Table 30: NBS Monograph 140

In [ ]:
def compare(table, y, factors):
    worst = 0.0
    print(f"{'statistic':>9} {'m':>4} {'published':>12} {'ntpstats':>14} {'rel. diff':>10}")
    for kind, published in table.items():
        r = nt.compute(to_phase(y), 1.0, kind, factors, ci=None, min_terms=1)
        for m, p, v in zip(factors, published, r.dev):
            rel = abs(v / p - 1)
            worst = max(worst, rel)
            print(f"{kind:>9} {m:>4} {p:12.7g} {v:14.9g} {rel:10.1e}")
    return worst


worst30 = compare(TABLE30, NBS9, [1, 2])
print(f"largest relative difference: {worst30:.1e}")

## Table 31: the 1000-point suite

In [ ]:
y = nist1000()
print(f"mean {y.mean():.7g} (published 4.897745e-01), std {y.std(ddof=1):.7g} (2.884664e-01)")
worst31 = compare(TABLE31, y, [1, 10, 100])
print(f"largest relative difference: {worst31:.1e}")
assert max(worst30, worst31) < 5e-7, "should agree to the 7 printed digits"

Every value agrees to the seven printed digits. The same values are frozen in the test suite, and
`docs/validation.md` lists the other references (analytic slopes, allantools, the EDF
approximations).